**MIRAGE DATASET** (copy from MIRAGE repo) and gathered with Claude Sonnet

| Column | Type | Notes |
|---|---|---|
| `source` | str | `popqa` / `naturalqa` / `triviaqa` / `ifqa` / `drop` |
| `query_id` | str | UUID — also stored as `oracle['mapped_id']` |
| `query` | str | Question text |
| `doc_name` | str | Oracle Wikipedia article title |
| `answer` | list[str] | One or more gold answer strings |
| `doc_url` | str | Wikipedia URL |
| `num_doc_labels` | int | Number of supporting chunks |
| `oracle` | dict | `{mapped_id, doc_name, doc_chunk, support}` — oracle passage, one per row |
| `doc_pool` | dict of lists | 5 candidate chunks per row: parallel lists `mapped_id`, `doc_name`, `doc_chunk`, `support` |

### Технические ячейки

In [1]:
!pip install -q datasets evaluate bert-score accelerate


[notice] A new release of pip is available: 23.0.1 -> 26.1.1
[notice] To update, run: python3 -m pip install --upgrade pip


In [50]:
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from bert_score import score as bert_score_fn
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM


import re
import ast
import string
import random
import warnings

from typing import List, Dict, Any

warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

SEED = 52
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

Device: cuda
GPU: Tesla V100-PCIE-32GB


In [16]:
hf_dataset = load_dataset("nlpai-lab/mirage")["train"]
print(hf_dataset)
print(f"Column names: {hf_dataset.column_names}")

Dataset({
    features: ['source', 'query_id', 'query', 'doc_name', 'answer', 'doc_url', 'num_doc_labels', 'doc_pool', 'oracle'],
    num_rows: 7560
})
Column names: ['source', 'query_id', 'query', 'doc_name', 'answer', 'doc_url', 'num_doc_labels', 'doc_pool', 'oracle']


Convert to a flat list of row-dicts — identical to `hf_dataset.to_list()` used in official code.

In [17]:
dataset: List[Dict[str, Any]] = hf_dataset.to_list()

row = dataset[0]
print("---")
print(row)

---
{'source': 'popqa', 'query_id': 'ce40d2c4-f403-4736-ace1-7fca9c722aba', 'query': "What is John Mayne's occupation?", 'doc_name': 'John Mayne', 'answer': ['journalist', 'journo', 'journalists'], 'doc_url': 'https://en.wikipedia.org/wiki?curid=1098597', 'num_doc_labels': 1, 'doc_pool': {'mapped_id': ['ce40d2c4-f403-4736-ace1-7fca9c722aba', 'ce40d2c4-f403-4736-ace1-7fca9c722aba', 'ce40d2c4-f403-4736-ace1-7fca9c722aba', 'ce40d2c4-f403-4736-ace1-7fca9c722aba', 'ce40d2c4-f403-4736-ace1-7fca9c722aba'], 'doc_name': ['John Mayne', 'John Mayne', 'John Dawson Mayne', 'John Dawson Mayne', 'John Dawson Mayne'], 'doc_chunk': ['Scottish printer, journalist and poet\nJohn Mayne (1759–1836) was a Scottish printer, journalist and poet born in Dumfries. In 1780, his poem "The Siller Gun" appeared in its original form in "Ruddiman\'s Magazine", published by Walter Ruddiman in Edinburgh. It is a humorous work on an ancient custom in Dumfries of shooting for the "Siller Gun." He also wrote a poem on "Ha

---
#### Основные метрики

**EM = exact matching**

| Metric | Normalisation | Rule |
|---|---|---|
| **EM-loose** | lowercase only | any gold answer is a *substring* of the prediction |
| **EM-strict** | lowercase only | any gold answer *exactly equals* the prediction |
| **F1 (MIRAGE)** | lowercase only | set-based token precision/recall harmonic mean |

In [18]:
def normalize_squad(text: str) -> str:
    text = text.lower()
    text = re.sub(r"\b(a|an|the)\b", " ", text)
    text = text.translate(str.maketrans("", "", string.punctuation))
    return " ".join(text.split())

def squad_em(prediction: str, golds: List[str]) -> float:
    np = normalize_squad(prediction)
    return float(any(normalize_squad(g) == np for g in golds))

def squad_f1(prediction: str, golds: List[str]) -> float:
    def _pair(pred, gold):
        pt = set(normalize_squad(pred).split())
        gt = set(normalize_squad(gold).split())
        tp = len(pt & gt)
        if not tp:
            return 0.0
        p, r = tp / len(pt), tp / len(gt)
        return 2 * p * r / (p + r)
    return max(_pair(prediction, g) for g in golds)

In [19]:
def em_loose(prediction: str, golds: List[str]) -> float:
    """
    то же самое, что и
    calculate_EM_loose() в файле оригинального репо LLM.py.
    """
    pred_lc = prediction.lower()
    return float(any(g.lower() in pred_lc for g in golds))

def em_strict(prediction: str, golds: List[str]) -> float:
    """
    то же самое, что и
    calculate_EM_strict() в файле оригинального репо LLM.py.
    """
    pred_lc = prediction.lower()
    return float(any(g.lower() == pred_lc for g in golds))

def mirage_f1(prediction: str, golds: List[str]) -> float:
    """
    то же самое, что и
    calculate_f1() в файле оригинального репо LLM.py.
    """
    pred_tokens = set(prediction.lower().split())
    gold_tokens = set(tok for g in golds for tok in g.lower().split())
    tp = len(pred_tokens & gold_tokens)
    if not tp:
        return 0.0
    p = tp / len(pred_tokens) if pred_tokens else 0.0
    r = tp / len(gold_tokens)  if gold_tokens  else 0.0
    return 2 * p * r / (p + r) if (p + r) else 0.0

Мы будем использовать функцию evaluate_answers для подсчета всех метрик (макро-усреднение).

In [45]:
def _to_gold_list(raw) -> List[str]:
    if isinstance(raw, list):
        return [str(v) for v in raw]
    if isinstance(raw, str):
        try:
            parsed = ast.literal_eval(raw)
            if isinstance(parsed, list):
                return [str(v) for v in parsed]
        except Exception:
            pass
        return [raw]
    return [str(raw)]

def evaluate_answers(
    predictions: List[str],
    references: List[Any],
    bertscore_model: str = "roberta-large",
    bs_batch_size: int = 32,
) -> Dict[str, float]:

    em_s, f1_s, loose_s, strict_s, mf1_s = [], [], [], [], []

    for pred, raw in zip(predictions, references):
        golds = _to_gold_list(raw)
        em_s.append(squad_em(pred, golds))
        f1_s.append(squad_f1(pred, golds))
        loose_s.append(em_loose(pred, golds))
        strict_s.append(em_strict(pred, golds))
        mf1_s.append(mirage_f1(pred, golds))

    flat_refs = [_to_gold_list(r)[0] for r in references]
    _, _, bsf1 = bert_score_fn(
        predictions, flat_refs,
        model_type=bertscore_model,
        lang="en",
        batch_size=bs_batch_size,
        verbose=False,
        device=DEVICE,
    )

    def _pct(lst): return round(sum(lst) / len(lst), 3)

    return {
        "EM (SQuAD-like)": _pct(em_s),
        "F1 (SQuAD-like)": _pct(f1_s),
        "EM-loose (MIRAGE)": _pct(loose_s),
        "EM-strict (MIRAGE)": _pct(strict_s),
        "F1 (MIRAGE)": _pct(mf1_s),
        "BERTScore-F1": round(bsf1.mean().item(), 3),
    }

---
## Task 2. Fine-tuned Encoder (MRC)

### 2.1 Extract questions, gold answers, and oracle passages

The oracle passage lives directly on each row at `row['oracle']['doc_chunk']`.
No secondary lookup is needed — this mirrors how `generate_LLM_prompt` uses
`self.oracle[data_dict['query_id']]['doc_chunk']` after building the dict from the same column.

In [21]:
questions = [row["query"] for row in dataset]
gold_answers = [_to_gold_list(row["answer"]) for row in dataset]
oracle_ctxs = [row["oracle"]["doc_chunk"] for row in dataset]

print(f"Всего строк: {len(questions)}")

Всего строк: 7560


In [22]:
print(f"{questions[0]}")

Вопросы What is John Mayne's occupation?


In [24]:
print(f"Голден ответы: {gold_answers[0]}")

Голден ответы: ['journalist', 'journo', 'journalists']


In [26]:
print(f"{oracle_ctxs[0]}")

Scottish printer, journalist and poet
John Mayne (1759–1836) was a Scottish printer, journalist and poet born in Dumfries. In 1780, his poem "The Siller Gun" appeared in its original form in "Ruddiman's Magazine", published by Walter Ruddiman in Edinburgh. It is a humorous work on an ancient custom in Dumfries of shooting for the "Siller Gun." He also wrote a poem on "Hallowe'en" in 1780 which influenced Robert Burns's 1785 poem "Halloween". Mayne also wrote a version of the ballad "Helen of Kirkconnel". His verses were admired by Walter Scott. Life. He was born at Dumfries on 26 March 1759. Educated at the local grammar school, he became a printer in the office of the "Dumfries Journal". In 1782 he went with his family to Glasgow, where he worked for five years in the publishing house of the brothers Foulis. In 1787 he settled in London, first as a printer, and then as proprietor and joint editor of "The Star", an evening paper, in which he placed his poems. He died at Lisson Grove, L

### QA pipeline from transformers

In [28]:
MRC_MODEL = "deepset/roberta-base-squad2"

mrc_pipe = pipeline(
    "question-answering",
    model=MRC_MODEL,
    device=0 if DEVICE == "cuda" else -1,
)
print(f"Loaded: {MRC_MODEL}")

Loaded: deepset/roberta-base-squad2


In [31]:
from tqdm.notebook import tqdm
MRC_BATCH = 16

mrc_inputs = [
    {"question": q, "context": c}
    for q, c in tqdm(zip(questions, oracle_ctxs), desc="input creation")
]
mrc_outputs = mrc_pipe(mrc_inputs, batch_size=MRC_BATCH, truncation=True, max_seq_len=512)

In [32]:
task2_preds = [out["answer"] for out in tqdm(mrc_outputs)]

  0%|          | 0/7560 [00:00<?, ?it/s]

In [44]:
print("Качество работы. Визуальная оценка\n")
for i in range(5):
    print(f"Q: {questions[i]}")
    print(f"Prediction: {task2_preds[i]}")
    print(f"Голден: {gold_answers[i]}")

Качество работы. Визуальная оценка

Q: What is John Mayne's occupation?
Prediction: printer
Голден: ['journalist', 'journo', 'journalists']
Q: What is Kathy Saltzman's occupation?
Prediction: American politician
Голден: ['politician', 'political leader', 'political figure', 'polit.', 'pol']
Q: What is Eleanor Davis's occupation?
Prediction: cartoonist and illustrator
Голден: ['cartoonist', 'graphic artist', 'animator', 'illustrator']
Q: What is William Murray, 1st Earl of Mansfield's occupation?
Prediction: lawyer
Голден: ['politician', 'political leader', 'political figure', 'polit.', 'pol']
Q: What is Þorsteinn Bachmann's occupation?
Prediction: actor
Голден: ['actor', 'actress', 'actors', 'actresses']


На первый взгляд, неплохо. В 3 из 4 случаев модель отвечает правильно. Что же будет по метрикам?

In [46]:
task2_scores = evaluate_answers(task2_preds, gold_answers)
for k, v in task2_scores.items():
    print(f"{k:<22}: {v}")

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: b9b1b17d-6a11-49d0-a4fc-793f8eea94d0)')' thrown while requesting HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


EM (SQuAD-like)       : 0.639
F1 (SQuAD-like)       : 0.753
EM-loose (MIRAGE)     : 0.721
EM-strict (MIRAGE)    : 0.615
F1 (MIRAGE)           : 0.639
BERTScore-F1          : 0.941


Всего 10 сломанных примеров из 7560

In [48]:
empty_indices = [i for i, p in enumerate(task2_preds) if not p.strip()]
print(f"Empty predictions: {len(empty_indices)} / {len(task2_preds)}")

for i in tqdm(empty_indices[:5]):
    print(f"idx={i}")
    print(f"Q: {questions[i]}")
    print(f"Gold: {gold_answers[i]}")
    print(f"Context: {oracle_ctxs[i][:1000]}\n")

Empty predictions: 10 / 7560


  0%|          | 0/5 [00:00<?, ?it/s]

idx=337
Q: In what city was Hittman born?
Gold: ['Los Angeles', 'Los Angeles, California', 'Pink City', 'The town of Our Lady the Queen of the Angels of the Little Portion', 'La La Land', 'Tinsel Town', 'City of Angels', 'City of Los Angeles', 'LA, California', 'L.A.', 'LA', 'Double Dubuque', 'Los Ángeles', 'Los Angeles, CA']
Context: Hittman may refer to:
Topics referred to by the same term
&lt;templatestyles src="Dmbox/styles.css" /&gt;
 This page lists associated with the title .

idx=1893
Q: Who was the screenwriter for The Ballad of Narayama?
Gold: ['Keisuke Kinoshita']
Context: The Ballad of Narayama may refer to:
Topics referred to by the same term
&lt;templatestyles src="Dmbox/styles.css" /&gt;
 This page lists associated with the title .

idx=1898
Q: Who was the screenwriter for The Stud?
Gold: ['Jackie Collins', 'Jacqueline Jill Collins']
Context: Stud may refer to:
&lt;templatestyles src="Template:TOC_right/styles.css" /&gt;
See also. Topics referred to by the same term
&lt;

---
## Task 1. 
*Choose a small (<=4B parameters) LM (e.g., Qwen/Gemma/Phi). Evaluate model’s answers on MIRAGE in base (0-shot) mode*

### Выберем Qwen 2.5 как индустриальный стандарт 
(тут можно взять инфу с поста Насти Казаковой про эту модель)

In [51]:
SMALL_LM = "Qwen/Qwen2.5-3B-Instruct"

lm_tok = AutoTokenizer.from_pretrained(SMALL_LM, trust_remote_code=True)

lm_model = AutoModelForCausalLM.from_pretrained(
    SMALL_LM,
    torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map="auto" if DEVICE == "cuda" else None,
    trust_remote_code=True,
)

lm_model.eval()

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 6780874b-5e3e-43f9-b11c-674fb3ecb7f2)')' thrown while requesting HEAD https://huggingface.co/Qwen/Qwen2.5-3B-Instruct/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
Loading checkpoint shards: 100%|██████████| 2/2 [00:14<00:00,  7.02s/it]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(151936, 2048)
    (layers): ModuleList(
      (0-35): 36 x Qwen2DecoderLayer(
        (self_attn): Qwen2SdpaAttention(
          (q_proj): Linear(in_features=2048, out_features=2048, bias=True)
          (k_proj): Linear(in_features=2048, out_features=256, bias=True)
          (v_proj): Linear(in_features=2048, out_features=256, bias=True)
          (o_proj): Linear(in_features=2048, out_features=2048, bias=False)
          (rotary_emb): Qwen2RotaryEmbedding()
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (up_proj): Linear(in_features=2048, out_features=11008, bias=False)
          (down_proj): Linear(in_features=11008, out_features=2048, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((2048,), eps=1e-06)
      )
    )
    (norm):

### Преобразование данных в диалоговый формат

Форматируем промпт для корректной обработки входа модели. Структура промпта взята из MIRAGE llm.py  `system_prompt` +
`base_format` / `RAG_format`

In [174]:
SYS = (
    """
    You are a question answering system.
    You should give short and concise answer on your prompt.
    You're STRICTLY PROHIBITED TO DO tool calling and add any reasoning to answer. 
    Output only the answer value, nothing else.
    No labels, no punctuation, no explanation.
    """
)

def make_base_prompt(row: Dict[str, Any]) -> str:
    msgs = [
        {"role": "system", "content": SYS},
        {"role": "user", "content": row["query"]},
        {"role": "assistant", "content": ""},
    ]
    return lm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

def make_oracle_prompt(row: Dict[str, Any]) -> str:
    ctx = row["oracle"]["doc_chunk"]
    msgs = [
        {"role": "system", "content": SYS},
        {"role": "user", "content": f"Context: {ctx}\n{row['query']}."},
        {"role": "assistant", "content": ""},
    ]
    return lm_tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)

print("====== BASE PROMPT ======:")
print(make_base_prompt(dataset[0]))
print("====== ORACLE PROMPT ======:")
print(make_oracle_prompt(dataset[0]))


====== BASE PROMPT ======:
<|im_start|>system

    You are a question answering system.
    You should give short and concise answer on your prompt.
    You're STRICTLY PROHIBITED TO DO tool calling and add any reasoning to answer. 
    Output only the answer value, nothing else.
    No labels, no punctuation, no explanation.
    <|im_end|>
<|im_start|>user
What is John Mayne's occupation? Answer: <|im_end|>
<|im_start|>assistant
<|im_end|>
<|im_start|>assistant

====== ORACLE PROMPT ======:
<|im_start|>system

    You are a question answering system.
    You should give short and concise answer on your prompt.
    You're STRICTLY PROHIBITED TO DO tool calling and add any reasoning to answer. 
    Output only the answer value, nothing else.
    No labels, no punctuation, no explanation.
    <|im_end|>
<|im_start|>user
Context: Scottish printer, journalist and poet
John Mayne (1759–1836) was a Scottish printer, journalist and poet born in Dumfries. In 1780, his poem "The Siller Gun" app

### Генерируем ответы

In [175]:
def generate_answers(
    prompts: List[str],
    tokenizer,
    model,
    batch_size: int = 8,
    max_new_tokens: int = 80,
) -> List[str]:

    results: List[str] = []

    for start in tqdm(range(0, len(prompts), batch_size), desc="Generating answer: "):
        batch = prompts[start:start + batch_size]

        enc = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=1024,
        ).to(model.device)

        with torch.no_grad():
            gen_ids = model.generate(
                **enc,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                pad_token_id=tokenizer.eos_token_id,
            )

        input_len = enc["input_ids"].shape[1]
        for ids in gen_ids:
            text = tokenizer.decode(ids[input_len:], skip_special_tokens=True)
            results.append(text.split("\n")[0].strip())

        done = min(start + batch_size, len(prompts))
        if done % (batch_size * 10) == 0 or done == len(prompts):
            print(f"{done}/{len(prompts)}")

    return results

### Инферим модель
и визуально оценим качество работы. Время работы - мое почтение.

In [164]:
base_prompts = [make_base_prompt(row) for row in dataset]

LM_BATCH = 8
MAX_NEW_TOKENS = 80

task1_preds = generate_answers(base_prompts, lm_tok, lm_model, LM_BATCH, MAX_NEW_TOKENS)

Generating answer:   0%|          | 0/945 [00:00<?, ?it/s]

80/7560
160/7560
240/7560
320/7560
400/7560
480/7560
560/7560
640/7560
720/7560
800/7560
880/7560
960/7560
1040/7560
1120/7560
1200/7560
1280/7560
1360/7560
1440/7560
1520/7560
1600/7560
1680/7560
1760/7560
1840/7560
1920/7560
2000/7560
2080/7560
2160/7560
2240/7560
2320/7560
2400/7560
2480/7560
2560/7560
2640/7560
2720/7560
2800/7560
2880/7560
2960/7560
3040/7560
3120/7560
3200/7560
3280/7560
3360/7560
3440/7560
3520/7560
3600/7560
3680/7560
3760/7560
3840/7560
3920/7560
4000/7560
4080/7560
4160/7560
4240/7560
4320/7560
4400/7560
4480/7560
4560/7560
4640/7560
4720/7560
4800/7560
4880/7560
4960/7560
5040/7560
5120/7560
5200/7560
5280/7560
5360/7560
5440/7560
5520/7560
5600/7560
5680/7560
5760/7560
5840/7560
5920/7560
6000/7560
6080/7560
6160/7560
6240/7560
6320/7560
6400/7560
6480/7560
6560/7560
6640/7560
6720/7560
6800/7560
6880/7560
6960/7560
7040/7560
7120/7560
7200/7560
7280/7560
7360/7560
7440/7560
7520/7560
7560/7560


In [168]:
print("Визуально оценим:\n")
for i in range(5):
    print(f"Q: {questions[i]}")
    print(f"Prediction: {task1_preds[i]}")
    print(f"Голден: {gold_answers[i]}")

Визуально оценим:

Q: What is John Mayne's occupation?
Prediction: Is this a typo? John Mayne is not a well-known public figure.
Голден: ['journalist', 'journo', 'journalists']
Q: What is Kathy Saltzman's occupation?
Prediction: Occupational Therapist
Голден: ['politician', 'political leader', 'political figure', 'polit.', 'pol']
Q: What is Eleanor Davis's occupation?
Prediction: None of the provided information indicates Eleanor Davis's occupation.
Голден: ['cartoonist', 'graphic artist', 'animator', 'illustrator']
Q: What is William Murray, 1st Earl of Mansfield's occupation?
Prediction: Judge
Голден: ['politician', 'political leader', 'political figure', 'polit.', 'pol']
Q: What is Þorsteinn Bachmann's occupation?
Prediction: None of the provided information indicates his occupation.
Голден: ['actor', 'actress', 'actors', 'actresses']


In [169]:
task1_scores = evaluate_answers(task1_preds, gold_answers)
for k, v in task1_scores.items():
    print(f"  {k:<22}: {v}")

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


  EM (SQuAD-like)       : 0.005
  F1 (SQuAD-like)       : 0.039
  EM-loose (MIRAGE)     : 0.012
  EM-strict (MIRAGE)    : 0.005
  F1 (MIRAGE)           : 0.032
  BERTScore-F1          : 0.814


In [170]:
empty_indices = [i for i, p in enumerate(task1_preds) if not p.strip()]
print(f"Empty predictions: {len(empty_indices)} / {len(task1_preds)}")

for i in tqdm(empty_indices[:5]):
    print(f"idx={i}")
    print(f"Q: {questions[i]}")
    print(f"Gold: {gold_answers[i]}")
    print(f"Context: {oracle_ctxs[i][:200]}\n")

Empty predictions: 23 / 7560


  0%|          | 0/5 [00:00<?, ?it/s]

idx=450
Q: What genre is Pre?
Gold: ['noise rock', 'noise punk']
Context: Pre, often written as PRE, is a British noise rock band, releasing music on the labels Skin Graft Records and Lovepump United. It is based in London and was formed around 2005. Pre includes former mem

idx=1102
Q: Who was the director of The Greatest?
Gold: ['Tom Gries', 'Thomas Stephen Gries', 'Monte Hellman', 'Monte Jay Himmelman', 'Monte Himmelbaum']
Context: 1977 film about Muhammad Ali
The Greatest is a 1977 biographical sports film about the life of boxer Muhammad Ali, in which Ali plays himself. It was directed by Tom Gries. The film follows Ali's life

idx=1739
Q: Who was the director of Tickets?
Gold: ['Ermanno Olmi', 'Abbas Kiarostami', 'Ken Loach', 'Kenneth Loach', 'Kenneth Charles Loach']
Context: Tickets is a 2005 comedy-drama anthology film directed by Ermanno Olmi, Abbas Kiarostami and Ken Loach. It was written by Ermanno Olmi, Abbas Kiarostami, and Paul Laverty. Three interconnected stories

idx=2

---
## Task 3.
Open-book question answering (MIRAGE’s oracle mode) with an instruction-tuned small LM.

Для работы в oracle mode нужно отформатировать промпты
### Причесывание промптов

Context is `row['oracle']['doc_chunk']`, injected directly -- same source the official
`RAG_format.format(query=..., context=self.oracle[query_id]['doc_chunk'])` uses.

In [176]:
oracle_prompts = [make_oracle_prompt(row) for row in dataset]

print(oracle_prompts[0])

<|im_start|>system

    You are a question answering system.
    You should give short and concise answer on your prompt.
    You're STRICTLY PROHIBITED TO DO tool calling and add any reasoning to answer. 
    Output only the answer value, nothing else.
    No labels, no punctuation, no explanation.
    <|im_end|>
<|im_start|>user
Context: Scottish printer, journalist and poet
John Mayne (1759–1836) was a Scottish printer, journalist and poet born in Dumfries. In 1780, his poem "The Siller Gun" appeared in its original form in "Ruddiman's Magazine", published by Walter Ruddiman in Edinburgh. It is a humorous work on an ancient custom in Dumfries of shooting for the "Siller Gun." He also wrote a poem on "Hallowe'en" in 1780 which influenced Robert Burns's 1785 poem "Halloween". Mayne also wrote a version of the ballad "Helen of Kirkconnel". His verses were admired by Walter Scott. Life. He was born at Dumfries on 26 March 1759. Educated at the local grammar school, he became a printer i

In [ ]:
task3_preds = generate_answers(oracle_prompts, lm_tok, lm_model, LM_BATCH, MAX_NEW_TOKENS)

Generating answer:   0%|          | 0/945 [00:00<?, ?it/s]

80/7560
160/7560
240/7560
320/7560
400/7560
480/7560
560/7560
640/7560
720/7560
800/7560
880/7560
960/7560
1040/7560
1120/7560
1200/7560
1280/7560
1360/7560
1440/7560
1520/7560
1600/7560
1680/7560
1760/7560
1840/7560
1920/7560
2000/7560
2080/7560
2160/7560
2240/7560
2320/7560
2400/7560
2480/7560
2560/7560
2640/7560
2720/7560
2800/7560
2880/7560
2960/7560
3040/7560
3120/7560
3200/7560
3280/7560
3360/7560
3440/7560
3520/7560
3600/7560
3680/7560
3760/7560
3840/7560
3920/7560
4000/7560
4080/7560
4160/7560
4240/7560
4320/7560
4400/7560
4480/7560
4560/7560
4640/7560
4720/7560
4800/7560
4880/7560
4960/7560
5040/7560
5120/7560
5200/7560
5280/7560
5360/7560
5440/7560
5520/7560
5600/7560
5680/7560
5760/7560
5840/7560
5920/7560
6000/7560
6080/7560
6160/7560
6240/7560
6320/7560
6400/7560
6480/7560
6560/7560
6640/7560
6720/7560
6800/7560
6880/7560
6960/7560
7040/7560
7120/7560
7200/7560
7280/7560
7360/7560
7440/7560
7520/7560
7560/7560


### Evaluate results of Task 3

In [ ]:
print("\n TASK 3. Qwen 2.5, but oracle passage injected):\n:")
for i in range(7):
    print(f"Q: {questions[i]}")
    print(f"Prediction: {task3_preds[i]}")
    print(f"Golden: {gold_answers[i]}")


 TASK 3. Qwen 2.5, but oracle passage injected):
:
Q: What is John Mayne's occupation?
Prediction: Life as a printer, journalist, and poet.
Golden: ['journalist', 'journo', 'journalists']
Q: What is Kathy Saltzman's occupation?
Prediction: Saltzman is an American politician and former member of the Minnesota Senate.
Golden: ['politician', 'political leader', 'political figure', 'polit.', 'pol']
Q: What is Eleanor Davis's occupation?
Prediction: Davis is an American cartoonist and illustrator.
Golden: ['cartoonist', 'graphic artist', 'animator', 'illustrator']
Q: What is William Murray, 1st Earl of Mansfield's occupation?
Prediction: Parliamentary politician, judge, lawyer, judge, peer
Golden: ['politician', 'political leader', 'political figure', 'polit.', 'pol']
Q: What is Þorsteinn Bachmann's occupation?
Prediction: Answerer
Golden: ['actor', 'actress', 'actors', 'actresses']
Q: What is Jacob Kraemer's occupation?
Prediction: Actor
Golden: ['actor', 'actress', 'actors', 'actresses']

In [ ]:
task3_scores = evaluate_answers(task3_preds, gold_answers)
print("Task 3")
for k, v in task3_scores.items():
    print(f"  {k:<22}: {v}")

'(ProtocolError('Connection aborted.', RemoteDisconnected('Remote end closed connection without response')), '(Request ID: 63481a2b-1c0e-4d84-85df-65c9884b09f4)')' thrown while requesting HEAD https://huggingface.co/roberta-large/resolve/main/tokenizer_config.json
Retrying in 1s [Retry 1/5].
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Task 3)
  EM (SQuAD-like)       : 0.328
  F1 (SQuAD-like)       : 0.529
  EM-loose (MIRAGE)     : 0.579
  EM-strict (MIRAGE)    : 0.318
  F1 (MIRAGE)           : 0.44
  BERTScore-F1          : 0.893


---
## Сравнение результатов работы моделей

In [ ]:
df = pd.DataFrame(
    {
        "0-shot Qwen 2.5": task1_scores,
        "": task2_scores,
        "Task 3 · ": task3_scores,
    }
).T
df

,EM (SQuAD-like),F1 (SQuAD-like),EM-loose (MIRAGE),EM-strict (MIRAGE),F1 (MIRAGE),BERTScore-F1
Task 1 ·,0.005,0.039,0.012,0.005,0.032,0.814
Task 2 ·,0.639,0.753,0.721,0.615,0.639,0.941
Task 3 ·,0.328,0.529,0.579,0.318,0.440,0.893


### Вывод: контекст улучшает качество модели

In [ ]:
print("Δ (Task 3 − Task 1)")
for metric in task1_scores:
    d = round(task3_scores[metric] - task1_scores[metric], 2)
    sign = "+" if d >= 0 else "-"
    print(f"  {metric:<22}: {sign}{d}")

Δ (Task 3 − Task 1)
  EM (SQuAD-like)       : +0.32
  F1 (SQuAD-like)       : +0.49
  EM-loose (MIRAGE)     : +0.57
  EM-strict (MIRAGE)    : +0.31
  F1 (MIRAGE)           : +0.41
  BERTScore-F1          : +0.08
